# Adverse Event Signal Exploration (FAERS/openFDA)

**Purpose:** reproducible descriptive exploration of de-identified adverse-event reports. This notebook queries a bounded sample of reports received in 2025 from the official openFDA API.

> **Caution:** spontaneous reports are unverified and may be incomplete or duplicated. Multiple drugs and reactions can appear in one report without a validated one-to-one link. Counts are not incidence rates and do not establish causation. Educational use only; not for clinical decisions.

Run top to bottom. Up to 3,000 reports are fetched and saved locally for repeatable analysis.

In [ ]:
from pathlib import Path
import json, time, requests
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
from IPython.display import display

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)
SNAPSHOT = DATA_DIR / "openfda_faers_2025_sample.json"
BASE_URL = "https://api.fda.gov/drug/event.json"
DATE_QUERY = "receivedate:[20250101 TO 20251231]"
MAX_RECORDS, PAGE_SIZE = 3000, 1000

## 1. Retrieve or reuse the snapshot

This fixed date window and record cap keep the download manageable. The API may update historical data. The local JSON snapshot is reused on later runs.

In [ ]:
if SNAPSHOT.exists():
    reports = json.loads(SNAPSHOT.read_text(encoding="utf-8"))
else:
    reports = []
    for skip in range(0, MAX_RECORDS, PAGE_SIZE):
        response = requests.get(BASE_URL, params={"search": DATE_QUERY, "limit": PAGE_SIZE, "skip": skip, "sort": "receivedate:asc"}, timeout=60)
        if response.status_code == 404:
            break
        response.raise_for_status()
        batch = response.json().get("results", [])
        reports.extend(batch)
        print(f"Fetched {len(reports):,} reports")
        if len(batch) < PAGE_SIZE:
            break
        time.sleep(0.25)
    SNAPSHOT.write_text(json.dumps(reports), encoding="utf-8")
if not reports:
    raise RuntimeError("No reports returned; check internet access and the openFDA API.")
print(f"Loaded {len(reports):,} reports")

## 2. Clean fields

Each row is one report. Terms are deduplicated per report before frequency counts. Age is converted to years when the submitted unit is usable; missing ages are retained as Unknown.

In [ ]:
def age_years(patient):
    try:
        n = float(patient.get("patientonsetage"))
        unit = str(patient.get("patientonsetageunit", ""))
        factor = {"800":1, "801":1/12, "802":1/52, "803":1/365, "804":1/8760, "805":10}
        return n * factor[unit] if unit in factor else None
    except (TypeError, ValueError):
        return None

def unique_terms(patient, section, key):
    return sorted({str(x.get(key, "")).strip().upper() for x in (patient.get(section, []) or []) if str(x.get(key, "")).strip()})

rows = []
for report in reports:
    p = report.get("patient", {}) or {}
    age = age_years(p)
    group = ("0-17" if age < 18 else "18-44" if age < 45 else "45-64" if age < 65 else "65+" if age <= 120 else "Unknown") if age is not None else "Unknown"
    rows.append({"report_id": report.get("safetyreportid"), "receivedate": report.get("receivedate"),
        "serious": str(report.get("serious", "")) == "1", "age_group": group,
        "drugs": unique_terms(p, "drug", "medicinalproduct"),
        "reactions": unique_terms(p, "reaction", " reactionmeddrapt".strip()),
        "outcomes": sorted({str(x.get("patientoutcome", "")) for x in (p.get("patientoutcome", []) or []) if x.get("patientoutcome")})})
df = pd.DataFrame(rows)
df["receivedate"] = pd.to_datetime(df["receivedate"], format="%Y%m%d", errors="coerce")
print(f"Sample size: {len(df):,}; dates: {df.receivedate.min().date()} to {df.receivedate.max().date()}")
display(df.head())

## 3. Reporting frequency and serious-report proportions

Seriousness is the share of sampled reports marked serious. It is not a drug-specific risk estimate: there are no exposure denominators, multiple products can appear on a report, and suspect associations are not validated.

In [ ]:
def report_frequency(field):
    exploded = df[["serious", field]].explode(field).dropna(subset=[field])
    exploded = exploded[exploded[field].astype(str).str.len() > 0]
    return exploded.groupby(field).agg(reports=("serious", "size"), serious_reports=("serious", "sum")).assign(serious_share=lambda x: x.serious_reports/x.reports).sort_values("reports", ascending=False)

drug_freq, reaction_freq = report_frequency("drugs"), report_frequency("reactions")
print(f"Marked serious: {df.serious.sum():,}/{len(df):,} ({df.serious.mean():.1%})")
display(drug_freq.head(15).style.format({"serious_share":"{:.1%}"}))
display(reaction_freq.head(15).style.format({"serious_share":"{:.1%}"}))

age_summary = df.groupby("age_group").agg(reports=("report_id", "size"), serious_reports=("serious", "sum"))
age_summary["serious_share"] = age_summary.serious_reports / age_summary.reports
display(age_summary.style.format({"serious_share":"{:.1%}"}))
outcome_counts = Counter(o for items in df.outcomes for o in items)
display(pd.DataFrame(outcome_counts.most_common(10), columns=["reported_outcome_code", "reports"]))

## 4. Visualize frequent terms and co-reports

These charts show sample counts, not incidence or comparative safety. Drug/reaction pairs mean only that both terms appeared in a report.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 7))
drug_freq.head(12).sort_values("reports").reports.plot.barh(ax=axes[0], color="#337ab7")
axes[0].set_title("Frequent reported drug names"); axes[0].set_xlabel("Reports in sample")
reaction_freq.head(12).sort_values("reports").reports.plot.barh(ax=axes[1], color="#58a68a")
axes[1].set_title("Frequent reported reaction terms"); axes[1].set_xlabel("Reports in sample")
plt.tight_layout(); plt.show()

pairs = Counter()
for _, row in df[["drugs", "reactions"]].iterrows():
    for drug in row.drugs:
        for reaction in row.reactions:
            pairs[(drug, reaction)] += 1
pair_table = pd.DataFrame([(d,e,n) for (d,e),n in pairs.most_common(20)], columns=["reported_drug_name","reaction_term","reports_with_both"])
display(pair_table)
if not pair_table.empty:
    pivot = pair_table.pivot(index="reported_drug_name", columns="reaction_term", values="reports_with_both").fillna(0)
    fig, ax = plt.subplots(figsize=(12, max(5, len(pivot)*.35)))
    im = ax.imshow(pivot.values, aspect="auto", cmap="Blues")
    ax.set_yticks(range(len(pivot.index)), labels=pivot.index)
    ax.set_xticks(range(len(pivot.columns)), labels=pivot.columns, rotation=60, ha="right")
    ax.set_title("Top co-reported drug–reaction pairs")
    fig.colorbar(im, ax=ax, label="Reports with both terms"); plt.tight_layout(); plt.show()

## 5. Generated descriptive summary

Use these data-driven results with the limitations above. Avoid causal wording.

In [ ]:
top_drugs = ", ".join(f"{name} ({int(r.reports)})" for name,r in drug_freq.head(5).iterrows())
top_reactions = ", ".join(f"{name} ({int(r.reports)})" for name,r in reaction_freq.head(5).iterrows())
print(f"In this bounded sample of {len(df):,} reports received in 2025, {df.serious.mean():.1%} were marked serious. Most frequent product names: {top_drugs}. Most frequent reaction terms: {top_reactions}. These are reporting frequencies only, not incidence, comparative risk, or proof of causation.")

## Sources and limitations

- Official source: https://open.fda.gov/apis/drug/event/
- API: https://api.fda.gov/drug/event.json
- Reports may be incomplete, duplicated, unverified, or delayed; historical records can be updated.
- Co-reported products and reactions do not establish a causal pairing. No exposure denominators are available here.
- This is a date-sorted cap of 3,000 records, not a random or complete sample. Results apply to this downloaded sample only.
- Educational exploration only. Do not use for medical care or clinical/product safety decisions.